# Qwen3-4B legal QLoRA — full dataset

This notebook trains **a fresh adapter** from the pinned Qwen3-4B base model. It consumes
`prepare_full_sft.py` output for all four domains. The dataset loader checks the manifest,
file hashes, and split isolation before model weights are loaded.

Run cells in order. `RUN_TRAINING=False` is the deliberate stop while preparation is in
progress. After inspecting the completed manifest and sample examples, change it to `True`,
restart the kernel, and run from the top. The notebook does not resume any pilot or leaked run.

**Edit guide:** The configuration cell contains the settings likely to change. After changing
`MAX_LENGTH`, tokenizer revision, or the source-selection policy, prepare a *new* dataset
version; this notebook will reject a mismatch. A GPU OOM requires a new preparation at a
shorter length. No cell silently truncates an example.

## 1. Configuration

`DATA_DIR` must point to a **complete** full-data preparation. `RUN_TRAINING` gates every
GPU cell. `RESUME_CHECKPOINT` is only for a checkpoint created by *this full dataset* with
matching run settings; leave it `None` for the first run. The old pilot adapter must not be
used as a resume checkpoint.

In [1]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src/legal_rag").is_dir():
    raise RuntimeError("Open this notebook with MyLLMAgent as the working directory.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Change v2 if your successful preparation used another NEW output directory.
DATA_DIR = PROJECT_ROOT / "data/training/legal-qa-full-v2"
TOKENIZER_DIR = PROJECT_ROOT / "data/training/qwen3-4b-tokenizer"
MODEL_ID = "Qwen/Qwen3-4B"
MODEL_REVISION = "1cfa9a7208912126459214e8b04321603b3df60c"
DOMAINS = ("civil", "criminal", "administrative", "intellectual_property")
MAX_LENGTH = 1024  # Must equal the preparation manifest's max_length.
SEED = 42

RUN_TRAINING = True  # Set True only when data preparation is complete and GPU is free.
RESUME_CHECKPOINT = None  # Example: Path("models/adapters/qwen3-4b-legal-full/<run>/checkpoints/checkpoint-25")

# Validation is a fixed sample for timely monitoring. The test remains separate.
VALIDATION_PER_DOMAIN = 32
TEST_PER_DOMAIN = 100  # Set None to evaluate every test example after saving.
GENERATIONS_PER_DOMAIN = 3
MAX_NEW_TOKENS = 384

print("Full-data directory:", DATA_DIR)
print("GPU training enabled:", RUN_TRAINING)

Full-data directory: /home/layke/Documents/MyLLMAgent/data/training/legal-qa-full-v2
GPU training enabled: True


## 2. Load and verify the completed dataset (CPU)

This call verifies the `complete` manifest, hashes, token lengths, and group/question
separation across train, validation, and test. The datasets are **disk-backed**; reading a
sample does not load all encoded examples into RAM. Preparation may take a long time, but
this notebook does not interfere with the running preparation process.

In [2]:
from legal_rag.training.dataset import file_hash
from legal_rag.training.full_sft import load_full_splits

raw_splits, tokenized, manifest = load_full_splits(DATA_DIR)
if manifest["max_length"] != MAX_LENGTH:
    raise RuntimeError("MAX_LENGTH differs from the prepared data; use a matching dataset.")
if manifest["tokenizer_revision"] != MODEL_REVISION:
    raise RuntimeError("Tokenizer/model revision mismatch; do not reuse these token IDs.")
if tuple(manifest["domains"]) != DOMAINS:
    raise RuntimeError("Prepared domains differ from this run's four domains.")
if any(len(tokenized[split]) == 0 for split in ("train", "validation", "test")):
    raise RuntimeError("A dataset split is empty.")

for split in ("train", "validation", "test"):
    counts = {domain: manifest["counts"][domain][split] for domain in DOMAINS}
    if sum(counts.values()) != len(tokenized[split]):
        raise RuntimeError(f"Manifest count mismatch in {split}.")
    print(f"{split}: {len(tokenized[split]):,} examples", counts)
print("Exclusions and source hashes: see", DATA_DIR / "manifest.json")
print("Rows needing review: see", DATA_DIR / "review.jsonl")

train: 65,448 examples {'civil': 13990, 'criminal': 19415, 'administrative': 21027, 'intellectual_property': 11016}
validation: 10,069 examples {'civil': 1675, 'criminal': 3427, 'administrative': 3608, 'intellectual_property': 1359}
test: 6,929 examples {'civil': 1655, 'criminal': 1909, 'administrative': 1927, 'intellectual_property': 1438}
Exclusions and source hashes: see /home/layke/Documents/MyLLMAgent/data/training/legal-qa-full-v2/manifest.json
Rows needing review: see /home/layke/Documents/MyLLMAgent/data/training/legal-qa-full-v2/review.jsonl


## 3. Verify tokenizer and inspect evidence (CPU)

The tokenizer hash must match the one used during preparation. `labels=-100` masks the
source and question, so loss is computed only for the complete answer and end-of-turn token.
Inspect examples from every domain; lexical screening is a triage check, not legal
verification. If an answer is unsupported, address the preparation data before training.

In [3]:
from transformers import AutoTokenizer

for filename, expected_hash in manifest["tokenizer_files"].items():
    if file_hash(TOKENIZER_DIR / filename) != expected_hash:
        raise RuntimeError(f"Tokenizer file changed since preparation: {filename}")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, local_files_only=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# One small example per domain. The full dataset stays on disk.
for domain in DOMAINS:
    example = next((row for row in raw_splits["train"] if row["domain"] == domain), None)
    if example is None:
        raise RuntimeError(f"No training example for {domain}")
    print("\nDOMAIN:", domain, "DOCUMENT:", example["document_id"])
    print("QUESTION:", example["question"])
    print("SOURCE:", example["context"])
    print("ANSWER:", example["answer"])
    print("Lexical coverage:", example["answer_lexical_coverage"])

sample = tokenized["train"][0]
supervised = [token for token, label in zip(sample["input_ids"], sample["labels"])
              if label != -100]
assert sample["input_ids"][-1] == tokenizer.eos_token_id
assert len(sample["input_ids"]) <= MAX_LENGTH
print("\nSupervised text:", tokenizer.decode(supervised))

/home/layke/anaconda3/envs/legal-qlora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



DOMAIN: civil DOCUMENT: 부산지방법원-2021노639
QUESTION: 반의사불벌죄에서 피해자가 처벌을 원하지 않는다는 의사표시의 법적 효력은 무엇인가요?
SOURCE:  과실로 피해자 소유의 위 (차량번호 3 생략) 오토바이를 쇼바 삼발이 교정 등 수리비 합계 1,520,000원 상당이 들도록 손괴하였다.
2) 판단
살피건대, 이 부분 도로교통법위반의 점은 도로교통법 제151조에 해당하는 죄로서 교통사고처리 특례법 제3조 제2항 본문에 따라 피해자의 명시적인 의사에 반하여 공소를 제기할 수 없다.
그런데 원심 소송기록에 의하면, 피해자 G이 이 사건 공소제기 후인 2020. 11. 17.경 원심법원에 피고인의 처벌을 원하지 않는다는 내용의 합의서를 제출한 사실을 인정할 수 있으므로, 원심으로서는 형사소송법 제327조 제6호에 의하여 이 부분 공소를 기각하였어야 할 것이다. 그럼에도 원심은 이를 그대로 유죄로 인정하였는바, 이러한 원심판결에는 반의사불벌죄에 있어서 피해자의 처벌불원 의사표시에 관한 사실을 오인하고 법리를 오해하여 판결을 그르친 잘못이 있다.
다.

[발췌 구분]

원심 소송기록에 의하면, 피해자 O가 이 사건 공소제기 후인 2021. 1. 12.경 원심법원에 피고인의 처벌을 원하지 않는다는 내용의 합의서를 제출한 사실을 인정할 수 있으므로, 원심으로서는 형사소송법 제327조 제6호에 의하여 이 부분 공소를 기각하였어야 할 것이다. 그럼에도 원심은 이를 그대로 유죄로 인정하였는바, 이러한 원심판결에는 반의사불벌죄에 있어서 피해자의 처벌불원 의사표시에 관한 사실을 오인하고 법리를 오해하여 판결을 그르친 잘못이 있다.
라. 소결
따라서 원심판결 중 2020고단2248 도로교통법위반의 점과 2020고단4485 도로교통법위반의 점에 관한 부분은 더 이상 유지할 수 없게 되었는바, 원심은 위와 같이 공소를 기각하였어야 할 위 각 공소사실 부분과 나머지 원심 판시 각 죄를 형법 제37조 전단의 경합범으로 처리하여 하나의 형을 선고하였으므

## 4. Select monitoring examples (CPU)

Training uses **every eligible training example** without domain downsampling. Validation
and test are distinct held-out splits. For timely monitoring, this notebook selects a fixed
subset per domain. Set `TEST_PER_DOMAIN=None` to evaluate the complete test split, which may
be much slower. Selection is based on example IDs, never on answer content or model score.

In [4]:
from hashlib import sha256
from torch.utils.data import Subset


def fixed_domain_indices(split, per_domain):
    # Iterate raw rows without retaining their context text. IDs are stable across runs.
    buckets = {domain: [] for domain in DOMAINS}
    for index, row in enumerate(raw_splits[split]):
        buckets[row["domain"]].append((row["id"], index))
    selected = []
    for domain in DOMAINS:
        ordered = sorted(buckets[domain], key=lambda pair: sha256(
            f"{SEED}:{pair[0]}".encode("utf-8")).hexdigest())
        chosen = ordered if per_domain is None else ordered[:per_domain]
        if not chosen:
            raise RuntimeError(f"No {split} rows available for {domain}.")
        selected.extend(index for _, index in chosen)
    return sorted(selected)

validation_indices = fixed_domain_indices("validation", VALIDATION_PER_DOMAIN)
test_indices = fixed_domain_indices("test", TEST_PER_DOMAIN)
validation_dataset = Subset(tokenized["validation"], validation_indices)
test_dataset = Subset(tokenized["test"], test_indices)

print("Training examples (all eligible):", len(tokenized["train"]))
print("Validation examples used for monitoring:", len(validation_dataset))
print("Test examples selected for final comparison:", len(test_dataset))

Training examples (all eligible): 65448
Validation examples used for monitoring: 128
Test examples selected for final comparison: 400


## 5. GPU boundary and memory check

Stop here until preparation finishes. Restarting the kernel before training releases prior
model objects. The memory check is an early warning; it cannot prove that every training
step will fit. Finish other GPU work and unload unused serving models before continuing.

In [5]:
if not RUN_TRAINING:
    raise RuntimeError("Preparation/review only. Set RUN_TRAINING=True and restart the kernel when ready.")

import importlib.metadata
import torch

required = {"torch": "2.8.0", "transformers": "4.57.1", "peft": "0.17.1",
            "bitsandbytes": "0.48.1", "accelerate": "1.10.1"}
for package, expected in required.items():
    actual = importlib.metadata.version(package).split("+")[0]
    if actual != expected:
        raise RuntimeError(f"Use env-qlora.yml: {package} {actual} != {expected}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in this notebook kernel.")
free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f"free {free/2**30:.2f} / {total/2**30:.2f} GiB")
if free < 6 * 2**30:
    raise RuntimeError("Less than 6 GiB is free; unload unused GPU processes first.")

NVIDIA GeForce RTX 4060 Laptop GPU free 7.51 / 7.62 GiB


## 6. Load 4-bit Qwen3-4B and attach a new LoRA adapter

NF4 quantization reduces base-weight memory. LoRA rank 8 adapts linear layers while the
base stays frozen. The adapter is initialized fresh here, including when resuming; Trainer
restores its weights and optimizer from a matching checkpoint in the next cell. The old
pilot/leaked adapter is never loaded. This cell can download base model weights.

In [6]:
if not RUN_TRAINING:
    raise RuntimeError("GPU work is disabled.")

from datetime import datetime, timezone
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(SEED)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, quantization_config=quantization,
    torch_dtype=compute_dtype, device_map={"": 0}, attn_implementation="sdpa",
)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(base_model, LoraConfig(
    task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules="all-linear", bias="none",
))
model.print_trainable_parameters()

# A fresh run gets a new artifact directory. Resuming points to the SAME run directory.
if RESUME_CHECKPOINT is None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    OUTPUT_DIR = PROJECT_ROOT / "models/adapters/qwen3-4b-legal-full" / stamp
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
else:
    RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT).resolve()
    if not (RESUME_CHECKPOINT / "trainer_state.json").is_file():
        raise FileNotFoundError(f"Not a Trainer checkpoint: {RESUME_CHECKPOINT}")
    OUTPUT_DIR = RESUME_CHECKPOINT.parent.parent

run_config = {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "manifest_sha256": file_hash(DATA_DIR / "manifest.json"),
    "dataset_files": manifest["files"], "max_length": MAX_LENGTH,
    "seed": SEED, "lora_r": 8, "lora_alpha": 16, "lora_dropout": 0.05,
    "validation_per_domain": VALIDATION_PER_DOMAIN,
    "train_examples": len(tokenized["train"]),
    "versions": {name: importlib.metadata.version(name) for name in required},
}
config_path = OUTPUT_DIR / "run_config.json"
if config_path.exists():
    if json.loads(config_path.read_text()) != run_config:
        raise RuntimeError("Checkpoint run config differs from this dataset/model/settings.")
else:
    if RESUME_CHECKPOINT is not None:
        raise RuntimeError("Checkpoint run_config.json is missing; cannot validate resume.")
    config_path.write_text(json.dumps(run_config, indent=2) + "\n")
print("Run artifacts:", OUTPUT_DIR)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:05<00:00,  1.82s/it]


trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089
Run artifacts: /home/layke/Documents/MyLLMAgent/models/adapters/qwen3-4b-legal-full/20260926T121806189251Z


## 7. Baseline validation, full training, and immediate adapter save

The complete training split is used. A fixed validation subset measures loss before and
after. Periodic checkpoints contain optimizer and scheduler state for an exact resume.
An OOM stops training; it does **not** cut tokens from the question, source or answer.

The trained adapter is saved immediately after `trainer.train()` and **before** post-training
evaluation. An OOM during later evaluation cannot erase the exported adapter.

In [7]:
if not RUN_TRAINING:
    raise RuntimeError("GPU work is disabled.")

import gc
from transformers import Trainer, TrainingArguments


def collate(features):
    # Pad only within this batch. Never truncate a prepared example.
    from legal_rag.training.tokenization import pad_batch
    padded = pad_batch(features, pad_token_id=tokenizer.pad_token_id)
    return {key: torch.tensor(value, dtype=torch.long) for key, value in padded.items()}


def save_run_metrics():
    (OUTPUT_DIR / "metrics.json").write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1, learning_rate=1e-4,
    lr_scheduler_type="cosine", warmup_ratio=0.03,
    optim="paged_adamw_8bit", weight_decay=0.01, max_grad_norm=1.0,
    bf16=compute_dtype == torch.bfloat16, fp16=compute_dtype == torch.float16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="no", prediction_loss_only=True,
    save_strategy="steps", save_steps=25, save_total_limit=2,
    save_only_model=False, logging_steps=10, report_to="none",
    seed=SEED, data_seed=SEED,
    dataloader_num_workers=0, dataloader_pin_memory=False,
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=tokenized["train"],
    eval_dataset=validation_dataset, data_collator=collate,
    processing_class=tokenizer,
)
(OUTPUT_DIR / "training_args.json").write_text(training_args.to_json_string())
metrics = {"train_examples": len(tokenized["train"]),
           "validation_examples": len(validation_dataset)}

# On resume, skip a new baseline: the adapter has not yet been restored here.
if RESUME_CHECKPOINT is None:
    print("Evaluating the untouched base model...")
    with model.disable_adapter():
        metrics["baseline_validation"] = trainer.evaluate(
            metric_key_prefix="base_validation")
    save_run_metrics()

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
try:
    train_result = trainer.train(
        resume_from_checkpoint=str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None)
except torch.cuda.OutOfMemoryError:
    metrics["training"] = {"status": "interrupted: CUDA OOM",
                           "global_step": trainer.state.global_step}
    save_run_metrics()
    print("Stopped after OOM. Restart the kernel; use a checkpoint from this run only.")
    raise

metrics["training"] = train_result.metrics
metrics["peak_allocated_gib"] = torch.cuda.max_memory_allocated() / 2**30
metrics["peak_reserved_gib"] = torch.cuda.max_memory_reserved() / 2**30
save_run_metrics()

# Export learned weights BEFORE any more evaluation or generation.
ADAPTER_DIR = OUTPUT_DIR / "adapter"
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(ADAPTER_DIR)
trainer.save_state()
if not (ADAPTER_DIR / "adapter_model.safetensors").is_file():
    raise RuntimeError("Adapter export is incomplete.")
metrics["adapter_directory"] = str(ADAPTER_DIR)
save_run_metrics()
print("Saved trained adapter:", ADAPTER_DIR)

# Release optimizer/scheduler memory before another validation pass.
trainer.optimizer = None
trainer.lr_scheduler = None
trainer.accelerator.free_memory()
del trainer
gc.collect()
torch.cuda.empty_cache()

inference_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "evaluation"),
    per_device_eval_batch_size=1, prediction_loss_only=True,
    bf16=compute_dtype == torch.bfloat16, fp16=compute_dtype == torch.float16,
    report_to="none", dataloader_num_workers=0, dataloader_pin_memory=False,
)
evaluation_trainer = Trainer(
    model=model, args=inference_args, eval_dataset=validation_dataset,
    data_collator=collate, processing_class=tokenizer,
)
try:
    metrics["adapter_validation"] = evaluation_trainer.evaluate(
        metric_key_prefix="adapter_validation")
except torch.cuda.OutOfMemoryError:
    metrics["adapter_validation"] = {"status": "CUDA OOM after adapter save"}
finally:
    save_run_metrics()
print(json.dumps(metrics, indent=2))

Evaluating the untouched base model...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.314200
20,1.171600
30,1.026300
40,0.730000
50,0.674100
60,0.639900
70,0.591100
80,0.606000
90,0.600200
100,0.617800


Saved trained adapter: /home/layke/Documents/MyLLMAgent/models/adapters/qwen3-4b-legal-full/20260924T105613457564Z/adapter


{
  "train_examples": 65448,
  "validation_examples": 128,
  "baseline_validation": {
    "base_validation_loss": 1.4525177478790283,
    "base_validation_model_preparation_time": 0.006,
    "base_validation_runtime": 63.703,
    "base_validation_samples_per_second": 2.009,
    "base_validation_steps_per_second": 2.009
  },
  "training": {
    "train_runtime": 128227.4782,
    "train_samples_per_second": 0.51,
    "train_steps_per_second": 0.032,
    "total_flos": 1.2977021614138737e+18,
    "train_loss": 0.4744288650578317,
    "epoch": 1.0
  },
  "peak_allocated_gib": 6.206376075744629,
  "peak_reserved_gib": 7.12109375,
  "adapter_directory": "/home/layke/Documents/MyLLMAgent/models/adapters/qwen3-4b-legal-full/20260924T105613457564Z/adapter",
  "adapter_validation": {
    "adapter_validation_loss": 0.4751080572605133,
    "adapter_validation_model_preparation_time": 0.0045,
    "adapter_validation_runtime": 76.5836,
    "adapter_validation_samples_per_second": 1.671,
    "adapter_v

## 8. Held-out test loss and paired answers

Only run after training and adapter export. The test sample is fixed by ID and never used
for optimizer updates. Per-domain loss is useful, but it does not establish legal accuracy.
Paired generations use the same prompt and decoding settings. Check whether the answer is
supported by the supplied passage and whether it ends cleanly. Prior inspected test rows
are not a pristine independent benchmark.

In [8]:
import gc
from pathlib import Path

import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)

from legal_rag.training.tokenization import pad_batch

OUTPUT_DIR = (
    PROJECT_ROOT
    / "models/adapters/qwen3-4b-legal-full"
    / "20260924T105613457564Z"
)
adapter_dir = OUTPUT_DIR / "adapter"

if not (adapter_dir / "adapter_model.safetensors").is_file():
    raise FileNotFoundError(adapter_dir)

set_seed(SEED)

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    quantization_config=quantization,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    attn_implementation="sdpa",
)

# Load the finished adapter for evaluation; no optimizer is created.
model = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    is_trainable=False,
)
model.eval()


def collate(features):
    padded = pad_batch(
        features,
        pad_token_id=tokenizer.pad_token_id,
    )
    return {
        key: torch.tensor(value, dtype=torch.long)
        for key, value in padded.items()
    }


evaluation_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "evaluation-reload"),
    per_device_eval_batch_size=1,
    prediction_loss_only=True,
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)

evaluation_trainer = Trainer(
    model=model,
    args=evaluation_args,
    data_collator=collate,
    processing_class=tokenizer,
)

print("Loaded saved adapter:", adapter_dir)

Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.54s/it]


Loaded saved adapter: /home/layke/Documents/MyLLMAgent/models/adapters/qwen3-4b-legal-full/20260924T105613457564Z/adapter


In [9]:
if not RUN_TRAINING:
    raise RuntimeError("GPU work is disabled.")
if not (OUTPUT_DIR / "adapter" / "adapter_model.safetensors").is_file():
    raise RuntimeError("Train and save the adapter before running the final comparison.")

from contextlib import nullcontext
from torch.utils.data import Subset
from legal_rag.training.tokenization import prompt_text

# comparison_dir = OUTPUT_DIR / "comparison"
comparison_dir = OUTPUT_DIR / "comparison-rerun-1"
comparison_dir.mkdir(exist_ok=True)
metrics_path = comparison_dir / "test_metrics.json"
answers_path = comparison_dir / "test_comparisons.jsonl"
# A failed comparison can be rerun in a NEW directory; never overwrite partial results.
if metrics_path.exists() or answers_path.exists():
    raise FileExistsError(f"Comparison output exists: {comparison_dir}")

model.eval()
model.gradient_checkpointing_disable()
per_domain = {}
for domain in DOMAINS:
    domain_indices = [i for i in test_indices if raw_splits["test"][i]["domain"] == domain]
    data = Subset(tokenized["test"], domain_indices)
    record = {"examples": len(data)}
    for label, enabled in (("base", False), ("adapter", True)):
        gc.collect()
        torch.cuda.empty_cache()
        context = nullcontext() if enabled else model.disable_adapter()
        try:
            with context:
                record[label] = evaluation_trainer.evaluate(
                    eval_dataset=data, metric_key_prefix=f"{label}_test")
        except torch.cuda.OutOfMemoryError:
            record[label] = {"status": "CUDA OOM"}
    per_domain[domain] = record
    metrics_path.write_text(json.dumps(per_domain, indent=2) + "\n")
    print(domain, record)


def generate_answer(example, *, use_adapter):
    set_seed(SEED)  # Identical random seed for each base/adapter pair.
    inputs = tokenizer(prompt_text(example, tokenizer), return_tensors="pt",
                       add_special_tokens=False).to("cuda:0")
    context = nullcontext() if use_adapter else model.disable_adapter()
    with context, torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
            temperature=0.7, top_p=0.8, top_k=20,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id, use_cache=True)
    completion = output[0, inputs["input_ids"].shape[1]:].detach().cpu().tolist()
    return {"text": tokenizer.decode(completion, skip_special_tokens=True),
            "generated_tokens": len(completion),
            "hit_length_limit": len(completion) == MAX_NEW_TOKENS and
                                completion[-1] != tokenizer.eos_token_id}

with answers_path.open("w", encoding="utf-8") as handle:
    for domain in DOMAINS:
        ids = [i for i in test_indices if raw_splits["test"][i]["domain"] == domain]
        for index in ids[:GENERATIONS_PER_DOMAIN]:
            row = raw_splits["test"][index]
            result = {key: row[key] for key in
                      ("id", "domain", "document_id", "question", "context", "answer")}
            for label, enabled in (("base", False), ("adapter", True)):
                gc.collect()
                torch.cuda.empty_cache()
                try:
                    result[label] = generate_answer(row, use_adapter=enabled)
                except torch.cuda.OutOfMemoryError:
                    result[label] = {"status": "CUDA OOM"}
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")
            handle.flush()  # Preserve completed pairs if a later pair fails.
print("Per-domain loss:", metrics_path)
print("Paired answers:", answers_path)

civil {'examples': 100, 'base': {'base_test_loss': 1.5052976608276367, 'base_test_model_preparation_time': 0.0051, 'base_test_runtime': 43.6008, 'base_test_samples_per_second': 2.294, 'base_test_steps_per_second': 2.294}, 'adapter': {'adapter_test_loss': 0.5984203815460205, 'adapter_test_model_preparation_time': 0.0051, 'adapter_test_runtime': 48.6369, 'adapter_test_samples_per_second': 2.056, 'adapter_test_steps_per_second': 2.056}}
criminal {'examples': 100, 'base': {'base_test_loss': 1.150941252708435, 'base_test_model_preparation_time': 0.0051, 'base_test_runtime': 43.0204, 'base_test_samples_per_second': 2.324, 'base_test_steps_per_second': 2.324}, 'adapter': {'adapter_test_loss': 0.35933375358581543, 'adapter_test_model_preparation_time': 0.0051, 'adapter_test_runtime': 48.1441, 'adapter_test_samples_per_second': 2.077, 'adapter_test_steps_per_second': 2.077}}
administrative {'examples': 100, 'base': {'base_test_loss': 1.194838523864746, 'base_test_model_preparation_time': 0.0051

## What the outputs mean

`OUTPUT_DIR/adapter/` is the usable PEFT adapter. `OUTPUT_DIR/checkpoints/` contains
resumable training state. Do not combine either with a pilot or a different base model.
The comparison uses the QA's known source, so it measures answer generation with supplied
evidence. Review groundedness and incorrect claims manually before any app integration.